In [1]:
# The following code will only execute
# successfully when compression is complete

import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohabenloughmari/miccai-task2-full")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/datasets/mohabenloughmari/miccai-task2-full


In [2]:
!pip install torch==2.5.0 torchvision==0.20.0 torchaudio==2.5.0 --index-url https://download.pytorch.org/whl/cu124

Looking in indexes: https://download.pytorch.org/whl/cu124
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 908.2/908.2 MB 1.6 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 7.2 MB/s eta 0:00:00:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 36.2 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 75.1 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 47.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 100.6 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.9 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.1 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 8.7 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 34.2 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12

In [3]:
!git clone https://github.com/Omid-Nejati/MedViTV2.git

Cloning into 'MedViTV2'...
remote: Enumerating objects: 189, done.
remote: Counting objects: 100% (77/77), done.
remote: Compressing objects: 100% (51/51), done.
remote: Total 189 (delta 54), reused 28 (delta 26), pack-reused 112 (from 1)
Receiving objects: 100% (189/189), 3.87 MiB | 32.18 MiB/s, done.
Resolving deltas: 100% (66/66), done.


In [4]:
!pip install natten==0.17.3+torch250cu124 -f https://shi-labs.com/natten/wheels/ --trusted-host shi-labs.com

Looking in links: https://shi-labs.com/natten/wheels/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 474.5/474.5 MB 1.4 MB/s eta 0:00:0000:0100:01


In [5]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA version: {torch.version.cuda}")

PyTorch version: 2.5.0+cu124
CUDA version: 12.4


In [ ]:
import pandas as pd
import torch
from tqdm import tqdm
import torch.nn as nn
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import os
import numpy as np
from sklearn.metrics import f1_score, cohen_kappa_score, recall_score, accuracy_score, classification_report
from scipy.stats import spearmanr
from torch.utils.data import WeightedRandomSampler

import warnings
warnings.filterwarnings("ignore")

base_path = path
path = os.path.join(base_path, 'Task_2')

df_train = pd.read_csv(os.path.join(path, "df_task2_train.csv"))
df_val = pd.read_csv(os.path.join(path, "df_task2_val.csv"))
df_test = pd.read_csv(os.path.join(path, "df_task2_test.csv"))

device = torch.device("cuda" if torch.cuda.is_available() else 'cpu')


class CustomImageDataset(Dataset):
    def __init__(self, dataframe, root_dir, transform=None):
        self.dataframe = dataframe
        self.root_dir = root_dir
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        img_name = os.path.join(self.root_dir, self.dataframe.iloc[idx]['image'])
        localiser_name = os.path.join(self.root_dir, self.dataframe.iloc[idx]['LOCALIZER'])

        image = Image.open(img_name).convert('RGB')
        localiser = Image.open(localiser_name).convert('RGB')

        if self.transform:
            image = self.transform(image)
            localiser = self.transform(localiser)

        label = self.dataframe.iloc[idx]['label']
        return image, localiser, label


train_transform = transforms.Compose([
    transforms.Resize((336, 336)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((336, 336)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
train_ds = CustomImageDataset(df_train, os.path.join(path, 'train'), transform=train_transform)
val_ds = CustomImageDataset(df_val, os.path.join(path, 'val'), transform=test_transform)
test_ds = CustomImageDataset(df_test, os.path.join(path, 'test'), transform=test_transform)

labels = df_train['label'].values.astype(int)

class_counts = np.bincount(labels)
sample_weights = 1.0 / class_counts[labels]
sampler = WeightedRandomSampler(sample_weights, len(sample_weights))
train_loader = DataLoader(train_ds, batch_size=12, sampler=sampler)
val_loader = DataLoader(val_ds, batch_size=12, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=12, shuffle=False)

n_labels = df_train['label'].nunique()





In [7]:
%cd /kaggle/working/MedViTV2
!pip install -r requirements.txt
from MedViT import MedViT

/kaggle/working/MedViTV2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 38.3 kB/s eta 0:00:00 0:00:01
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 63.9 MB/s eta 0:00:0000:0100:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 2.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.2/61.2 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.7/178.7 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.8/58.8 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━

In [8]:



class DualBranchMedViT(nn.Module):
    def __init__(self, num_classes, pretrained_weights=None, **kwargs):
        super().__init__()
        
        # Two separate MedViT encoders (or shared weights — your choice)
        self.branch_image = MedViT(num_classes=num_classes, **kwargs)
        self.branch_localizer = MedViT(num_classes=num_classes, **kwargs)
        
        # Remove both classification heads — we'll fuse before classifying
        final_dim = self.branch_image.proj_head[0].in_features  # e.g. 512
        self.branch_image.proj_head = nn.Identity()
        self.branch_localizer.proj_head = nn.Identity()
        
        # Fusion + classification head
        self.classifier = nn.Sequential(
            nn.Linear(final_dim * 2, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )
        
        if pretrained_weights:
            state_dict = torch.load(pretrained_weights, map_location='cpu')
            self.branch_image.load_state_dict(state_dict, strict=False)
            self.branch_localizer.load_state_dict(state_dict, strict=False)

    def forward(self, image, localizer):
        feat_img = self.branch_image(image)        # (B, final_dim)
        feat_loc = self.branch_localizer(localizer) # (B, final_dim)
        fused = torch.cat([feat_img, feat_loc], dim=1)  # (B, final_dim*2)
        return self.classifier(fused)

In [9]:
NUM_CLASSES = df_train['label'].nunique()

model = DualBranchMedViT(
    num_classes=NUM_CLASSES,
    stem_chs=[64, 32, 64],
    depths=[2, 2, 6, 2],       # MedViT-Small (depths[2] must be divisible by 3)
    dims=[64, 128, 320, 512],
    path_dropout=0.2,
).to(device)


initialize_weights...
initialize_weights...


In [10]:
optimizer = torch.optim.AdamW([
    {'params': model.branch_image.parameters(), 'lr': 1e-3},
    {'params': model.branch_localizer.parameters(), 'lr': 1e-3},
    {'params': model.classifier.parameters(), 'lr': 1e-3},  # much higher for the head
], weight_decay=0.05)

In [11]:
import torch.nn.functional as F
class FocalLoss(nn.Module):
    def __init__(self, gamma=2):
        super().__init__()
        self.gamma = gamma
        
    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = (1 - pt) ** self.gamma * ce_loss
        return focal_loss.mean()

criterion = FocalLoss(gamma=2)

In [12]:
from sklearn.metrics import f1_score, matthews_corrcoef, confusion_matrix, cohen_kappa_score, classification_report
import numpy as np

def specificity(class_ground_truth, class_prediction):
    eps = 0.000001
    cnf_matrix = confusion_matrix(class_ground_truth, class_prediction)
    FP = cnf_matrix.sum(axis=0) - np.diag(cnf_matrix)
    FN = cnf_matrix.sum(axis=1) - np.diag(cnf_matrix)
    TP = np.diag(cnf_matrix)
    TN = cnf_matrix.sum() - (FP + FN + TP)
    FP, FN, TP, TN = FP.astype(float), FN.astype(float), TP.astype(float), TN.astype(float)
    spe = TN / (TN + FP + eps)
    return spe.mean()

def evaluate(model, dataloader, criterion, device, show_report=True):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        pbar = tqdm(dataloader, desc='Evaluation', leave=False)
        for images, localisers, labels in pbar:
            images, localisers, labels = images.to(device), localisers.to(device), labels.long().to(device)
            outputs = model(images, localisers)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    accuracy = 100. * correct / total
    
    # Print classification report if requested
    if show_report:
        print("\n" + "="*80)
        print("CLASSIFICATION REPORT - Per-Class Performance:")
        print("="*80)
        print(classification_report(all_labels, all_preds, 
                                    target_names=['Class 0', 'Class 1', 'Class 2'],
                                    digits=4))
        print("="*80 + "\n")
    
    return {
        'loss': total_loss / len(dataloader),
        'accuracy': accuracy,
        'F1_score': f1_score(all_labels, all_preds, average='micro'),
        'Rk-correlation': matthews_corrcoef(all_labels, all_preds),
        'Specificity': specificity(all_labels, all_preds),
        'Quadratic-weighted_Kappa': cohen_kappa_score(all_labels, all_preds, weights='quadratic')
    }

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

num_epochs = 5
for epoch in range(num_epochs):
    model.train()
    all_preds = []
    all_labels = []
    train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Train]")
    for images, localizers, labels in train_bar:
        images = images.to(device)
        localizers = localizers.to(device)
        labels = labels.to(device).long()
        outputs = model(images, localizers)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_bar.set_postfix(loss=f"{loss.item():.4f}")

        _, predicted = outputs.max(1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    print(f"\nEpoch {epoch+1}/{num_epochs} — "
          f"Acc: {acc:.4f} | "
          f"F1: {f1_score(all_labels, all_preds, average='micro'):.4f} | "
          f"MCC: {matthews_corrcoef(all_labels, all_preds):.4f} | "
          f"Spec: {specificity(all_labels, all_preds):.4f} | "
          f"QWK: {cohen_kappa_score(all_labels, all_preds, weights='quadratic'):.4f}\n")
    
    # ADD THIS - Per-class performance
    print("Per-Class Performance:")
    print(classification_report(all_labels, all_preds, 
                                target_names=['Class 0', 'Class 1', 'Class 2'],
                                digits=4))
    print("-" * 80)

Epoch 1/5 [Train]: 100%|██████████| 674/674 [17:27<00:00,  1.55s/it, loss=0.5404]



Epoch 1/5 — Acc: 0.3471 | F1: 0.3471 | MCC: 0.0201 | Spec: 0.6733 | QWK: 0.0101

Per-Class Performance:
              precision    recall  f1-score   support

     Class 0     0.3553    0.3634    0.3593      2779
     Class 1     0.3430    0.3391    0.3411      2604
     Class 2     0.3422    0.3379    0.3400      2699

    accuracy                         0.3471      8082
   macro avg     0.3468    0.3468    0.3468      8082
weighted avg     0.3470    0.3471    0.3470      8082

--------------------------------------------------------------------------------


Epoch 2/5 [Train]:  36%|███▋      | 246/674 [06:20<11:15,  1.58s/it, loss=0.6687]

In [ ]:
results = evaluate(model, test_loader, criterion, device)
print(f"Accuracy: {results['accuracy']:.2f}%")
print(f"F1 Score: {results['F1_score']:.4f}")
print(f"Quadratic Weighted Kappa: {results['Quadratic-weighted_Kappa']:.4f}")
print(f"Rk Correlation (Spearman): {results['Rk-correlation']:.4f}")
print(f"Specificity: {results['Specificity']:.4f}")

In [ ]:
torch.save(model.state_dict(), 'model_weights.pt')
